In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# =====================================================
# STYLE
# =====================================================
sns.set_theme(style="whitegrid")
palette = sns.color_palette("Set2")

# =====================================================
# CREATE FOLDER FOR FIGURES
# =====================================================
os.makedirs("figures", exist_ok=True)

# =====================================================
# LOAD DATASET
# =====================================================
file_path = "pon_synthetic_dataset_50000_overlap.csv"
data = pd.read_csv(file_path)

print("=" * 60)
print("Dataset loaded successfully")
print("Shape:", data.shape)

# =====================================================
# FEATURES AND TARGET
# =====================================================
features = [
    "packet_size_bytes",
    "packets_per_sec",
    "traffic_rate_mbps"
]

X = data[features]
y = data["traffic_type"]

# =====================================================
# 1. FEATURE VISUALIZATION
# =====================================================

# -----------------------------------------------------
# 1.1 Traffic class distribution
# -----------------------------------------------------
plt.figure(figsize=(7, 5))
sns.countplot(
    data=data,
    x="traffic_type",
    palette="Set2",
    order=sorted(data["traffic_type"].unique())
)

plt.xlabel("Traffic Type")
plt.ylabel("Number of Samples")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/traffic_class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# 1.2 Scatter plot: packets_per_sec vs traffic_rate_mbps
# -----------------------------------------------------
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=data.sample(5000, random_state=42),   # sample for readability
    x="packets_per_sec",
    y="traffic_rate_mbps",
    hue="traffic_type",
    palette="Set2",
    alpha=0.6
)

plt.xlabel("Packets per Second")
plt.ylabel("Traffic Rate (Mbps)")
plt.legend(title="Traffic Type")

ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold')   
plt.tight_layout()
plt.savefig("figures/scatter_packets_vs_rate.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# 1.3 Scatter plot: packet_size_bytes vs traffic_rate_mbps
# -----------------------------------------------------
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=data.sample(5000, random_state=42),
    x="packet_size_bytes",
    y="traffic_rate_mbps",
    hue="traffic_type",
    palette="Set2",
    alpha=0.6
)

plt.xlabel("Packet Size (Bytes)")
plt.ylabel("Traffic Rate (Mbps)")
plt.legend(title="Traffic Type")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/scatter_packet_size_vs_rate.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# 1.4 Pairplot of core traffic features
# -----------------------------------------------------
sample_data = data.sample(2000, random_state=42)

pairplot = sns.pairplot(
    sample_data,
    vars=features,
    hue="traffic_type",
    palette="Set2",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 20}
)

pairplot.savefig("figures/pairplot_features.png", dpi=300, bbox_inches="tight")
plt.show()

# =====================================================
# 2. OVERFITTING ANALYSIS
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

models = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=7))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=6,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            C=0.3,
            gamma="scale",
            class_weight="balanced"
        ))
    ])
}

results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    train_f1 = f1_score(y_train, y_train_pred, average="weighted")
    test_f1 = f1_score(y_test, y_test_pred, average="weighted")

    results.append({
        "Model": model_name,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc,
        "Train F1-score": train_f1,
        "Test F1-score": test_f1,
        "Overfitting Gap": train_f1 - test_f1
    })

results_df = pd.DataFrame(results)
print("\nOverfitting Analysis:")
print(results_df)

results_df.to_csv("overfitting_analysis_results.csv", index=False)

# -----------------------------------------------------
# 2.1 Train vs Test performance
# -----------------------------------------------------
comparison_df = results_df.melt(
    id_vars="Model",
    value_vars=["Train F1-score", "Test F1-score"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=comparison_df,
    x="Model",
    y="Score",
    hue="Metric",
    palette="Set2"
)

plt.xlabel("Model")
plt.ylabel("F1-score")
plt.ylim(0, 1.05)
plt.legend(title="Metric")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/train_test_f1_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# 2.2 Overfitting gap
# -----------------------------------------------------
plt.figure(figsize=(7, 5))
sns.barplot(
    data=results_df,
    x="Model",
    y="Overfitting Gap",
    palette="Set2"
)

plt.xlabel("Model")
plt.ylabel("Overfitting Gap")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/overfitting_gap.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# 2.3 Learning curves
# -----------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def plot_learning_curve(estimator, X_data, y_data, model_name):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator,
        X_data,
        y_data,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5),
        shuffle=True,
        random_state=42
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, marker="o", label="Training Score", color=palette[0])
    plt.plot(train_sizes, val_mean, marker="o", label="Validation Score", color=palette[1])

    plt.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.2,
        color=palette[0]
    )
    plt.fill_between(
        train_sizes,
        val_mean - val_std,
        val_mean + val_std,
        alpha=0.2,
        color=palette[1]
    )

    plt.xlabel("Training Set Size")
    plt.ylabel("Weighted F1-score")
    plt.legend()
    ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
    ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
    plt.xticks(fontsize=11, color='#333333', fontweight='bold')
    plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
    plt.tight_layout()
    plt.savefig(f"figures/learning_curve_{model_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

for model_name, model in models.items():
    plot_learning_curve(model, X_train, y_train, model_name.replace(" ", "_"))

print("\nSaved files:")
print("- figures/traffic_class_distribution.png")
print("- figures/scatter_packets_vs_rate.png")
print("- figures/scatter_packet_size_vs_rate.png")
print("- figures/pairplot_features.png")
print("- figures/train_test_f1_comparison.png")
print("- figures/overfitting_gap.png")
print("- figures/learning_curve_KNN.png")
print("- figures/learning_curve_Random_Forest.png")
print("- figures/learning_curve_SVM.png")
print("- overfitting_analysis_results.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'pon_synthetic_dataset_50000_overlap.csv'

In [5]:
import time
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
palette = sns.color_palette("Set2")

# =====================================================
# COMPUTATIONAL COST ANALYSIS
# =====================================================

complexity_results = []

for model_name, model in models.items():

    print(f"\nEvaluating computational cost for {model_name}")

    # ------------------------------
    # TRAINING TIME
    # ------------------------------
    start_train = time.perf_counter()
    model.fit(X_train, y_train)
    end_train = time.perf_counter()

    training_time = end_train - start_train

    # ------------------------------
    # INFERENCE TIME
    # ------------------------------
    start_pred = time.perf_counter()
    y_pred = model.predict(X_test)
    end_pred = time.perf_counter()

    inference_time = end_pred - start_pred

    # average time per sample
    inference_per_sample = inference_time / len(X_test)

    # ------------------------------
    # MODEL SIZE
    # ------------------------------
    model_bytes = pickle.dumps(model)
    model_size_kb = len(model_bytes) / 1024

    complexity_results.append({
        "Model": model_name,
        "Training Time (s)": training_time,
        "Inference Time (s)": inference_time,
        "Inference per Sample (ms)": inference_per_sample * 1000,
        "Model Size (KB)": model_size_kb
    })

complexity_df = pd.DataFrame(complexity_results)

print("\nComputational Complexity Results")
print(complexity_df)

complexity_df.to_csv("computational_cost_results.csv", index=False)

# =====================================================
# FIGURE 1: TRAINING TIME
# =====================================================

plt.figure(figsize=(7,5))

sns.barplot(
    data=complexity_df,
    x="Model",
    y="Training Time (s)",
    palette="Set2"
)

plt.xlabel("Model")
plt.ylabel("Training Time (seconds)")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/training_time_models.png", dpi=300, bbox_inches="tight")

plt.show()

# =====================================================
# FIGURE 2: INFERENCE TIME
# =====================================================

plt.figure(figsize=(7,5))

sns.barplot(
    data=complexity_df,
    x="Model",
    y="Inference per Sample (ms)",
    palette="Set2"
)

plt.xlabel("Model")
plt.ylabel("Inference Time per Sample (ms)")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/inference_time_models.png", dpi=300, bbox_inches="tight")

plt.show()

# =====================================================
# FIGURE 3: MODEL SIZE
# =====================================================

plt.figure(figsize=(7,5))

sns.barplot(
    data=complexity_df,
    x="Model",
    y="Model Size (KB)",
    palette="Set2"
)

plt.xlabel("Model")
plt.ylabel("Model Size (KB)")
ax.set_xlabel(var, fontsize=12, color='#111111', fontweight='bold')
ax.set_ylabel(kpi, fontsize=12, color='#111111', fontweight='bold')
plt.xticks(fontsize=11, color='#333333', fontweight='bold')
plt.yticks(fontsize=11, color='#333333', fontweight='bold') 
plt.tight_layout()
plt.savefig("figures/model_size_models.png", dpi=300, bbox_inches="tight")

plt.show()

print("\nSaved files:")
print("- computational_cost_results.csv")
print("- figures/training_time_models.png")
print("- figures/inference_time_models.png")
print("- figures/model_size_models.png")

SyntaxError: invalid syntax (3661746050.py, line 16)